### Bucket 1 — Refit champions on all data & save

Evaluation is done, so I refit the three champions (Ridge, LightGBM-balanced,
LogReg-balanced) on the **entire** feature table — no held-out test now, I want
every row of signal. I save the fitted **pipelines** (preprocessing baked in) to
`models/` via joblib, so the Predictor can just load and call `.predict()`.

In [1]:
# --- Session 8 · Bucket 1: refit champions on ALL data + persist ---
%load_ext autoreload
%autoreload 2

import pandas as pd
import joblib
from src import config
from src.models.trainer import ModelTrainer

# 1) full feature table
features = pd.read_parquet(config.FEATURES_PATH)

# 2) X + two targets (regression value, classification category)
X_all = features[config.MODEL_FEATURES]
y_reg = features[config.TARGET_AQI_COL]

# encode category -> severity-ordered int (Good=0. Severe=5), same as Session 7
cat_to_id = {name: i for i, name in enumerate(config.AQI_CATEGORIES)}
y_clf = features[config.TARGET_BUCKET_COL].map(cat_to_id)
assert y_clf.notna().all(), "unmapped category in target_bucket"

# 3) pull the exact champions (defaults won — tuning was a wash)
t = ModelTrainer()
reg        = dict(t.regression_roster())["Ridge"]
clf_lgbm   = dict(t.classification_roster(balanced=True))["LightGBM"]
clf_logreg = dict(t.classification_roster(balanced=True))["LogisticRegression"]

# 4) wrap each in the leakage-safe pipeline and fit on EVERYTHING
champions = {
    "ridge_regressor":   t.make_pipeline(reg).fit(X_all, y_reg),
    "lgbm_classifier":   t.make_pipeline(clf_lgbm).fit(X_all, y_clf),
    "logreg_classifier": t.make_pipeline(clf_logreg).fit(X_all, y_clf),
}

# 5) save fitted PIPELINES (preprocessing travels inside the model)
config.MODELS_DIR.mkdir(parents=True, exist_ok=True)
for name, pipe in champions.items():
    joblib.dump(pipe, config.MODELS_DIR / f"{name}.joblib")

print("saved:", sorted(p.name for p in config.MODELS_DIR.glob("*.joblib")))

saved: ['lgbm_classifier.joblib', 'logreg_classifier.joblib', 'ridge_regressor.joblib']


In [2]:
# reload from disk and predict on the most recent row
sample = X_all.tail(1)

reg_pipe = joblib.load(config.MODELS_DIR / "ridge_regressor.joblib")
clf_pipe = joblib.load(config.MODELS_DIR / "lgbm_classifier.joblib")

print("next-day AQI :", round(float(reg_pipe.predict(sample)[0]), 1))
print("next-day cat :", config.AQI_CATEGORIES[int(clf_pipe.predict(sample)[0])])

next-day AQI : 111.8
next-day cat : Moderate


### Bucket 2 — Predictor skeleton

The `Predictor` loads the saved pipelines once and holds the feature table so it
can look up any `city + date`. This bucket is just the plumbing: load models,
find the right feature row. The actual `.predict()` comes next.

In [3]:
# --- Session 8 · Bucket 2: Predictor skeleton ---
class Predictor:
    """Turn a city + date into its model-ready feature row (forecasts come next).

    Loads the fitted champion pipelines once and keeps the feature table in
    memory so any historical day can be looked up and fed to the models.

    Attributes:
        reg: Fitted regression pipeline (Ridge) -> next-day AQI value.
        clf: Fitted classification pipeline (LightGBM) -> next-day category.
        features: The full feature table used for city/date lookups.
    """

    def __init__(self, features: pd.DataFrame | None = None) -> None:
        """Load the saved pipelines and the feature table.

        Args:
            features: Optional pre-loaded feature frame; read from disk if None.
        """
        self.reg = joblib.load(config.MODELS_DIR / "ridge_regressor.joblib")
        self.clf = joblib.load(config.MODELS_DIR / "lgbm_classifier.joblib")
        if features is None:
            features = pd.read_parquet(config.FEATURES_PATH)
        self.features = features

    def _feature_row(self, city_id: str, date) -> pd.DataFrame:
        """Fetch the single feature row for one city on one date.

        Args:
            city_id: Canonical "city, state" identity.
            date: The day whose features drive the T+1 forecast.

        Returns:
            A one-row DataFrame of MODEL_FEATURES.

        Raises:
            ValueError: If no row exists for that city and date.
        """
        date = pd.Timestamp(date)
        mask = (
            (self.features[config.CITY_ID_COL] == city_id)
            & (self.features[config.DATE_COL] == date)
        )
        row = self.features.loc[mask]
        if row.empty:
            raise ValueError(f"no feature row for {city_id!r} on {date.date()}")
        return row[config.MODEL_FEATURES]

In [4]:
pred = Predictor()

# grab a real city+date straight from the table so we know it exists
last = pred.features.iloc[-1]
row = pred._feature_row(last[config.CITY_ID_COL], last[config.DATE_COL])

print("row shape:", row.shape)          # (1, 22)
print("city/date:", last[config.CITY_ID_COL], last[config.DATE_COL].date())

row shape: (1, 22)
city/date: Yamuna Nagar, Haryana 2023-03-30
